# Churn Labeling - Non-Contractual Setting

Phương pháp:
1. Tính IPT (Inter-Purchase Time) cho từng user
2. Tính Lapse Point = μ + 2σ của IPT
3. Gán nhãn Churn = 1 nếu Recency trong Target Window > Lapse Point

**Không data leakage**: nhãn chỉ dựa trên hành vi trong Target Window, không dùng thông tin từ Feature Window để gán nhãn.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = r'c:\Users\Admin\data_driven_marketing'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')

df = pd.read_pickle(os.path.join(DATA_DIR, 'df_full.pkl'))

# Chuẩn hóa cột thời gian
if 'visitStartTime_datetime' in df.columns:
    df['event_date'] = pd.to_datetime(df['visitStartTime_datetime'], errors='coerce')
elif 'date' in df.columns:
    df['event_date'] = pd.to_datetime(df['date'], errors='coerce')
else:
    df['event_date'] = pd.to_datetime(df['visitStartTime'], unit='s', errors='coerce')

df['fullVisitorId'] = df['fullVisitorId'].astype(str)

print('Shape:', df.shape)
print('Date range:', df['event_date'].min(), '->', df['event_date'].max())

In [ ]:
# ==============================================================
# BƯỚC 1: Tính IPT (Inter-Purchase Time)
# Chỉ tính trên toàn bộ lịch sử, KHÔNG dùng feature/target window
# Mục đích: xác định ngưỡng churn toàn cục
# ==============================================================

# Lấy 1 session/ngày mỗi user (unique visit per day)
visits = (
    df[['fullVisitorId', 'event_date']]
    .drop_duplicates()
    .sort_values(['fullVisitorId', 'event_date'])
    .copy()
)

# Tính khoảng cách giữa các lần visit liên tiếp
visits['prev_date'] = visits.groupby('fullVisitorId')['event_date'].shift(1)
visits['ipt_days'] = (visits['event_date'] - visits['prev_date']).dt.days

# Chỉ giữ các user có >= 2 lần visit mới tính được IPT
ipt_series = visits['ipt_days'].dropna()

mu = ipt_series.mean()
sigma = ipt_series.std()
lapse_point = mu + 2 * sigma

print('='*60)
print('IPT Statistics (toàn bộ lịch sử):')
print(f'  Số cặp visit có IPT: {len(ipt_series):,}')
print(f'  Mean IPT (μ):        {mu:.2f} ngày')
print(f'  Std IPT (σ):         {sigma:.2f} ngày')
print(f'  Lapse Point (μ+2σ):  {lapse_point:.2f} ngày')
print('='*60)
print()
print(f'Ý nghĩa: Nếu khách không quay lại sau {lapse_point:.0f} ngày,')
print('         xác suất họ đã churn > 95%.')

In [ ]:
# Visualize phân phối IPT
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution
axes[0].hist(ipt_series.clip(upper=200), bins=100, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(mu, color='orange', linestyle='--', linewidth=2, label=f'μ = {mu:.1f} days')
axes[0].axvline(lapse_point, color='red', linestyle='--', linewidth=2, label=f'Lapse = μ+2σ = {lapse_point:.1f} days')
axes[0].set_title('Phân phối IPT (clip tại 200 ngày)', fontsize=13)
axes[0].set_xlabel('Inter-Purchase Time (ngày)')
axes[0].set_ylabel('Số cặp visit')
axes[0].legend()

# Percentile view
pcts = np.arange(50, 100, 1)
vals = np.percentile(ipt_series, pcts)
axes[1].plot(pcts, vals, color='steelblue', linewidth=2)
axes[1].axhline(lapse_point, color='red', linestyle='--', linewidth=2, label=f'Lapse Point = {lapse_point:.1f} days')
axes[1].set_title('Percentile IPT (P50 - P99)', fontsize=13)
axes[1].set_xlabel('Percentile')
axes[1].set_ylabel('IPT (ngày)')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'ipt_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()
print(f'\nSaved: {os.path.join(DATA_DIR, "ipt_distribution.png")}')

In [ ]:
# ==============================================================
# BƯỚC 2: Thiết lập Time Windows (giống Train_test_split.ipynb)
# ==============================================================

FEATURE_DAYS = 168
TARGET_DAYS  = 92
STEP_DAYS    = 60

base_date = df['event_date'].min()
max_date  = df['event_date'].max()

print(f'Base date : {base_date.date()}')
print(f'Max date  : {max_date.date()}')
print(f'Lapse Point: {lapse_point:.2f} ngày')

In [ ]:
# ==============================================================
# BƯỚC 3: Hàm gán nhãn Churn cho từng fold
#
# Logic KHÔNG data leakage:
#   - Feature Window: [feature_start, feature_end)  -> tính features
#   - Target Window : [target_start, target_end)    -> tính nhãn
#
# Churn = 1 nếu:
#   Recency = (target_end - last_visit_in_target_window) > lapse_point
#   HOẶC user hoàn toàn không xuất hiện trong Target Window
#
# Recency đo từ CUỐI target window để biết:
#   "Tính đến lúc mô hình dự báo, user đã im bặt bao lâu?"
# ==============================================================

def label_churn_for_fold(df_raw, k, feature_days, target_days,
                          step_days, lapse_point_days, base_date):
    """
    Trả về DataFrame mức user với cột:
      - fullVisitorId
      - fold_id
      - feature_start, feature_end, target_start, target_end
      - last_visit_in_feature   : ngày visit cuối trong Feature Window
      - last_visit_in_target    : ngày visit cuối trong Target Window (NaT nếu không xuất hiện)
      - recency_days            : số ngày kể từ last visit in target đến target_end
      - appeared_in_target      : 1/0 - có xuất hiện trong Target Window
      - churn                   : NHÃN (1 = churn, 0 = active)
    """
    feature_start = base_date + pd.Timedelta(days=step_days * (k - 1))
    feature_end   = feature_start + pd.Timedelta(days=feature_days)
    target_start  = feature_end
    target_end    = target_start + pd.Timedelta(days=target_days)

    if target_end > df_raw['event_date'].max():
        print(f'Skip Fold {k}: target_end {target_end.date()} > max_date {df_raw["event_date"].max().date()}')
        return None

    # Users xuất hiện trong Feature Window
    df_feat = df_raw[
        (df_raw['event_date'] >= feature_start) &
        (df_raw['event_date'] <  feature_end)
    ]
    users_in_feature = df_feat['fullVisitorId'].unique()

    # Last visit của mỗi user trong Feature Window
    last_feat = (
        df_feat.groupby('fullVisitorId')['event_date']
        .max()
        .rename('last_visit_in_feature')
        .reset_index()
    )

    # -----------------------------------------------------------
    # TARGET WINDOW: chỉ dùng để gán nhãn, KHÔNG leak vào features
    # -----------------------------------------------------------
    df_tgt = df_raw[
        (df_raw['event_date'] >= target_start) &
        (df_raw['event_date'] <  target_end)
    ]

    # Chỉ xét users đã xuất hiện trong Feature Window
    df_tgt_feat_users = df_tgt[df_tgt['fullVisitorId'].isin(users_in_feature)]

    # Last visit của mỗi user trong Target Window
    last_tgt = (
        df_tgt_feat_users.groupby('fullVisitorId')['event_date']
        .max()
        .rename('last_visit_in_target')
        .reset_index()
    )

    # Merge
    fold_df = last_feat.merge(last_tgt, on='fullVisitorId', how='left')

    fold_df['appeared_in_target'] = fold_df['last_visit_in_target'].notna().astype(int)

    # Recency = target_end - last_visit_in_target
    # Nếu không xuất hiện trong Target Window -> recency = TARGET_DAYS (toàn bộ cửa sổ)
    fold_df['last_visit_for_recency'] = fold_df['last_visit_in_target'].fillna(target_start)
    fold_df['recency_days'] = (target_end - fold_df['last_visit_for_recency']).dt.days

    # NHÃN CHURN
    # Churn = 1 nếu recency > lapse_point_days
    fold_df['churn'] = (fold_df['recency_days'] > lapse_point_days).astype(int)

    # Metadata
    fold_df['fold_id']       = k
    fold_df['feature_start'] = feature_start
    fold_df['feature_end']   = feature_end
    fold_df['target_start']  = target_start
    fold_df['target_end']    = target_end

    n_users  = len(fold_df)
    n_churn  = fold_df['churn'].sum()
    n_active = n_users - n_churn
    print(f'Fold {k}: Feature {feature_start.date()} -> {feature_end.date()} | '
          f'Target {target_start.date()} -> {target_end.date()} | '
          f'Users={n_users:,} | Churn={n_churn:,} ({n_churn/n_users*100:.1f}%) | '
          f'Active={n_active:,} ({n_active/n_users*100:.1f}%)')

    return fold_df[[
        'fullVisitorId', 'fold_id',
        'feature_start', 'feature_end', 'target_start', 'target_end',
        'last_visit_in_feature', 'last_visit_in_target',
        'appeared_in_target', 'recency_days', 'churn'
    ]]


print('Hàm label_churn_for_fold đã sẵn sàng.')

In [ ]:
# ==============================================================
# BƯỚC 4: Chạy gán nhãn cho tất cả các fold
# ==============================================================

print(f'Lapse Point = {lapse_point:.2f} ngày\n')
print('='*90)

fold_labels = []
k = 1
while True:
    result = label_churn_for_fold(
        df_raw           = df,
        k                = k,
        feature_days     = FEATURE_DAYS,
        target_days      = TARGET_DAYS,
        step_days        = STEP_DAYS,
        lapse_point_days = lapse_point,
        base_date        = base_date
    )
    if result is None:
        break
    fold_labels.append(result)
    k += 1

print('='*90)
all_labels = pd.concat(fold_labels, ignore_index=True)
print(f'\nTổng số fold: {all_labels["fold_id"].nunique()}')
print(f'Tổng số records: {len(all_labels):,}')
print(f'Churn rate tổng: {all_labels["churn"].mean()*100:.2f}%')

In [ ]:
# ==============================================================
# BƯỚC 5: Kiểm tra phân bố nhãn theo fold
# ==============================================================

summary = (
    all_labels.groupby('fold_id')
    .agg(
        n_users          = ('fullVisitorId', 'count'),
        n_churn          = ('churn', 'sum'),
        churn_rate       = ('churn', 'mean'),
        appeared_in_tgt  = ('appeared_in_target', 'mean'),
        mean_recency     = ('recency_days', 'mean'),
        feature_start    = ('feature_start', 'first'),
        target_end       = ('target_end', 'first')
    )
    .reset_index()
)

summary['churn_rate_pct']      = (summary['churn_rate'] * 100).round(2)
summary['appeared_in_tgt_pct'] = (summary['appeared_in_tgt'] * 100).round(2)
summary['mean_recency']        = summary['mean_recency'].round(1)

display(summary[[
    'fold_id', 'n_users', 'n_churn', 'churn_rate_pct',
    'appeared_in_tgt_pct', 'mean_recency',
    'feature_start', 'target_end'
]])

In [ ]:
# ==============================================================
# BƯỚC 6: Visualize churn rate & recency distribution
# ==============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(summary['fold_id'], summary['churn_rate_pct'], color='salmon', edgecolor='white')
axes[0].axhline(all_labels['churn'].mean()*100, color='red', linestyle='--',
                linewidth=2, label=f'Overall churn = {all_labels["churn"].mean()*100:.1f}%')
axes[0].set_title('Churn Rate theo Fold', fontsize=13)
axes[0].set_xlabel('Fold ID')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].legend()

axes[1].hist(all_labels['recency_days'].clip(upper=TARGET_DAYS+5), bins=60,
             color='steelblue', alpha=0.8, edgecolor='white')
axes[1].axvline(lapse_point, color='red', linestyle='--', linewidth=2,
                label=f'Lapse Point = {lapse_point:.1f} days')
axes[1].set_title('Phân phối Recency trong Target Window', fontsize=13)
axes[1].set_xlabel('Recency (ngày)')
axes[1].set_ylabel('Số users')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'churn_label_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {os.path.join(DATA_DIR, "churn_label_distribution.png")}')

In [ ]:
# ==============================================================
# BƯỚC 7: Lưu kết quả
# ==============================================================

out_path = os.path.join(DATA_DIR, 'churn_labels.pkl')
all_labels.to_pickle(out_path)
print(f'Đã lưu: {out_path}')
print(f'Shape: {all_labels.shape}')
print()
print('Columns:')
print(all_labels.dtypes)
print()
print('Sample:')
display(all_labels.head(10))